# GameTheory 27b — Le lake `assignment_lean` par son certificat (compagnon natif)

**Navigation** : [<< 27-Munkres-Assignment (track principal)](GameTheory-27-Munkres-Assignment.ipynb) | [Index](README.md) | [`assignment_lean/`](assignment_lean/)

**Autruche pédagogique** : le notebook [GT-27](GameTheory-27-Munkres-Assignment.ipynb) *implémente* la méthode hongroise en Python (labels, arbre hongrois, `scipy.optimize.linear_sum_assignment`) et vérifie son triple test **numériquement**. Ce compagnon 27b exécute le **lake Lean `assignment_lean`** sous kernel natif `lean4-wsl` : les mêmes énoncés — dualité faible, certificat à gap nul, invariant de sortie, resserrement hongrois — deviennent des théorèmes **prouvés par le noyau**, et nous terminons sur une preuve complète qu'une affectation concrète 3×3 est optimale, sans énumérer les factorielles.

Hommage : James R. Munkres (1930–2026), co-éponyme de l'algorithme de Kuhn-Munkres (Kuhn 1955 ; Munkres 1957) — le récit historique est dans le [GT-27](GameTheory-27-Munkres-Assignment.ipynb).

---
## 1. Le lake importé — quatre modules, une chaîne

Le lake suit la convention i18n du dépôt (EPIC #4980) : chaque module français a son sibling `_en`. La chaîne d'imports est linéaire — `KuhnMunkres` → `Optimality` → `Duality` → `Definitions` — un seul import charge donc tout le lake.

In [1]:
import Assignment.KuhnMunkres

open Assignment

import Assignment.KuhnMunkres

open Assignment
--% env 0

Raw input:
{"cmd": "import Assignment.KuhnMunkres\n\nopen Assignment"}
Raw output:
{"env": 0}

### Lecture du résultat

L'import passe sans erreur : les `.olean` du lake (compilés par `lake build Assignment`) sont résolus par le kernel. Le lake expose **dix déclarations publiques**, que nous vérifions d'un coup — ces signatures SONT le plan du cours :

In [2]:
-- Module Definitions — le probleme primal
#check Assignment.value
#check Assignment.IsOptimal

-- Module Duality — le couple dual
#check Assignment.DualFeasible
#check Assignment.dualValue
#check Assignment.weak_duality

-- Module Optimality — le certificat
#check Assignment.dualValue_eq_of_edges
#check Assignment.optimality_of_zero_gap

-- Module KuhnMunkres — la charpente de correction
#check Assignment.EqEdge
#check Assignment.kuhn_munkres_correct
#check Assignment.dualFeasible_tighten

-- Module Definitions — le probleme primal
#check Assignment.value
──────▶  Assignment.value {n : ℕ} (C : Fin n → Fin n → ℤ) (σ : Equiv.Perm (Fin n)) : ℤ
#check Assignment.IsOptimal
──────▶  Assignment.IsOptimal {n : ℕ} (C : Fin n → Fin n → ℤ) (σ : Equiv.Perm (Fin n)) : Prop

-- Module Duality — le couple dual
#check Assignment.DualFeasible
──────▶  Assignment.DualFeasible {n : ℕ} (C : Fin n → Fin n → ℤ) (u v : Fin n → ℤ) : Prop
#check Assignment.dualValue
──────▶  Assignment.dualValue {n : ℕ} (u v : Fin n → ℤ) : ℤ
#check Assignment.weak_duality
──────▶  Assignment.weak_duality {n : ℕ} (C : Fin n → Fin n → ℤ) (u v : Fin n → ℤ) (h : DualFeasible C u v)
  (σ : Equiv.Perm (Fin n)) : dualValue u v ≤ value C σ

-- Module Optimality — le certificat
#check Assignment.dualValue_eq_of_edges
──────▶  Assignment.dualValue_eq_of_edges {n : ℕ} (C : Fin n → Fin n → ℤ) (u v : Fin n → ℤ) (σ : Equiv.Perm (Fin n))
  (h : ∀ (i : Fin n), u i + v (σ i) = C i (σ i)) : dualValue u v = value C σ
#check Assignment.optimality_of_zero_gap
──────▶  Assignment.optimality_of_zero_gap {n : ℕ} (C : Fin n → Fin n → ℤ) (u v : Fin n → ℤ) (σ : Equiv.Perm (Fin n))
  (h : DualFeasible C u v) (heq : dualValue u v = value C σ) : IsOptimal C σ

-- Module KuhnMunkres — la charpente de correction
#check Assignment.EqEdge
──────▶  Assignment.EqEdge {n : ℕ} (C : Fin n → Fin n → ℤ) (u v : Fin n → ℤ) (i j : Fin n) : Prop
#check Assignment.kuhn_munkres_correct
──────▶  Assignment.kuhn_munkres_correct {n : ℕ} (C : Fin n → Fin n → ℤ) (u v : Fin n → ℤ) (σ : Equiv.Perm (Fin n))
  (h : DualFeasible C u v) (heq : ∀ (i : Fin n), EqEdge C u v i (σ i)) : IsOptimal C σ
#check Assignment.dualFeasible_tighten
──────▶  Assignment.dualFeasible_tighten {n : ℕ} (C : Fin n → Fin n → ℤ) (u v : Fin n → ℤ) (S T : Finset (Fin n))
  (h : DualFeasible C u v) (δ : ℤ) (hδ : 0 ≤ δ) (hmargin : ∀ i ∈ S, ∀ j ∉ T, δ ≤ C i j - (u i + v j)) :
  DualFeasible C (fun i => if i ∈ S then u i + δ else u i) fun j => if j ∈ T then v j - δ else v j
--% env 1

Raw input:
{"cmd": "-- Module Definitions \u2014 le probleme primal\n#check Assignment.value\n#check Assignment.IsOptimal\n\n-- Module Duality \u2014 le couple dual\n#check Assignment.DualFeasible\n#check Assignment.dualValue\n#check Assignment.weak_duality\n\n-- Module Optimality \u2014 le certificat\n#check Assignment.dualValue_eq_of_edges\n#check Assignment.optimality_of_zero_gap\n\n-- Module KuhnMunkres \u2014 la charpente de correction\n#check Assignment.EqEdge\n#check Assignment.kuhn_munkres_correct\n#check Assignment.dualFeasible_tighten", "env": 0}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "Assignment.value {n : ℕ} (C : Fin n → Fin n → ℤ) (σ : Equiv.Perm (Fin n)) : ℤ"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "Assignment.IsOptimal {n : ℕ} (C : Fin n → Fin n → ℤ) (σ : Equiv.Perm (Fin n)) : Prop"},
  {"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 6},
   "data":
   "Assignment.DualFeasible {n : ℕ} (C : Fin n → Fin n → ℤ) (u v : Fin n → ℤ) : Prop"},
  {"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 6},
   "data": "Assignment.dualValue {n : ℕ} (u v : Fin n → ℤ) : ℤ"},
  {"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 6},
   "data":
   "Assignment.weak_duality {n : ℕ} (C : Fin n → Fin n → ℤ) (u v : Fin n → ℤ) (h : DualFeasible C u v)\n  (σ : Equiv.Perm (Fin n)) : dualValue u v ≤ value C σ"},
  {"severity": "info",
   "pos": {"line": 11, "column": 0},
   "endPos": {"line": 11, "column": 6},
   "data":
   "Assignment.dualValue_eq_of_edges {n : ℕ} (C : Fin n → Fin n → ℤ) (u v : Fin n → ℤ) (σ : Equiv.Perm (Fin n))\n  (h : ∀ (i : Fin n), u i + v (σ i) = C i (σ i)) : dualValue u v = value C σ"},
  {"severity": "info",
   "pos": {"line": 1

---
## 2. Le problème sur un exemple fil rouge : trois agents, trois tâches

Toute la suite travaille sur la **même matrice de coûts 3×3** — celle de la section 2 de GT-27 : l'agent $i$ coûte $C_{ij}$ à affecter à la tâche $j$, et on **minimise** le coût total. En Lean, la matrice est une fonction `Fin 3 → Fin 3 → ℤ` (arithmétique entière exacte, comme l'algorithme pédagogique de GT-27) :

In [3]:
-- La matrice des couts de GT-27 (section 2)
def C3 : Fin 3 → Fin 3 → ℤ := ![![4, 1, 3], ![2, 0, 5], ![3, 2, 2]]

#eval C3 0 1   -- cout d'affecter l'agent 0 a la tache 1
#eval C3 1 2   -- cout d'affecter l'agent 1 a la tache 2

-- La matrice des couts de GT-27 (section 2)
def C3 : Fin 3 → Fin 3 → ℤ := ![![4, 1, 3], ![2, 0, 5], ![3, 2, 2]]

#eval C3 0 1   -- cout d'affecter l'agent 0 a la tache 1
─────▶  1
#eval C3 1 2   -- cout d'affecter l'agent 1 a la tache 2
─────▶  5
--% env 2

Raw input:
{"cmd": "-- La matrice des couts de GT-27 (section 2)\ndef C3 : Fin 3 \u2192 Fin 3 \u2192 \u2124 := ![![4, 1, 3], ![2, 0, 5], ![3, 2, 2]]\n\n#eval C3 0 1   -- cout d'affecter l'agent 0 a la tache 1\n#eval C3 1 2   -- cout d'affecter l'agent 1 a la tache 2", "env": 1}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 5},
   "data": "1"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 5},
   "data": "5"}],
 "env": 2}

---
## 3. La valeur d'une affectation (`Assignment.value`)

Une affectation est un **matching parfait** : chaque agent reçoit exactement une tâche, chaque tâche un agent — c'est une permutation `Equiv.Perm (Fin 3)`. Sa valeur est la somme des coûts des arêtes empruntées (`Assignment.value` : $\sum_i C_{i,\sigma(i)}$). Il y a $3! = 6$ affectations ; évaluons-les toutes :

In [4]:
-- Les 6 permutations de Fin 3, de la plus simple aux 3-cycles
#eval Assignment.value C3 (Equiv.refl _)                       -- identite
#eval Assignment.value C3 (Equiv.swap (0 : Fin 3) 1)           -- transposition 0<->1
#eval Assignment.value C3 (Equiv.swap (0 : Fin 3) 2)           -- transposition 0<->2
#eval Assignment.value C3 (Equiv.swap (1 : Fin 3) 2)           -- transposition 1<->2
#eval Assignment.value C3 ((Equiv.swap (0 : Fin 3) 1).trans (Equiv.swap (0 : Fin 3) 2))
#eval Assignment.value C3 ((Equiv.swap (0 : Fin 3) 1).trans (Equiv.swap (0 : Fin 3) 2)).symm

-- Les 6 permutations de Fin 3, de la plus simple aux 3-cycles
#eval Assignment.value C3 (Equiv.refl _)                       -- identite
─────▶  6
#eval Assignment.value C3 (Equiv.swap (0 : Fin 3) 1)           -- transposition 0<->1
─────▶  5
#eval Assignment.value C3 (Equiv.swap (0 : Fin 3) 2)           -- transposition 0<->2
─────▶  6
#eval Assignment.value C3 (Equiv.swap (1 : Fin 3) 2)           -- transposition 1<->2
─────▶  11
#eval Assignment.value C3 ((Equiv.swap (0 : Fin 3) 1).trans (Equiv.swap (0 : Fin 3) 2))
─────▶  9
#eval Assignment.value C3 ((Equiv.swap (0 : Fin 3) 1).trans (Equiv.swap (0 : Fin 3) 2)).symm
─────▶  7
--% env 3

Raw input:
{"cmd": "-- Les 6 permutations de Fin 3, de la plus simple aux 3-cycles\n#eval Assignment.value C3 (Equiv.refl _)                       -- identite\n#eval Assignment.value C3 (Equiv.swap (0 : Fin 3) 1)           -- transposition 0<->1\n#eval Assignment.value C3 (Equiv.swap (0 : Fin 3) 2)           -- transposition 0<->2\n#eval Assignment.value C3 (Equiv.swap (1 : Fin 3) 2)           -- transposition 1<->2\n#eval Assignment.value C3 ((Equiv.swap (0 : Fin 3) 1).trans (Equiv.swap (0 : Fin 3) 2))\n#eval Assignment.value C3 ((Equiv.swap (0 : Fin 3) 1).trans (Equiv.swap (0 : Fin 3) 2)).symm", "env": 2}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data": "6"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "5"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 5},
   "data": "6"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 5},
   "data": "11"},
  {"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 5},
   "data": "9"},
  {"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 5},
   "data": "7"}],
 "env": 3}

### Lecture du résultat

Le minimum est **5**, atteint par la transposition $\sigma^* = (0 \leftrightarrow 1)$ : l'agent 0 prend la tâche 1 (coût 1), l'agent 1 la tâche 0 (coût 2), l'agent 2 la tâche 2 (coût 2). Ici l'énumération suffit — 6 cas. Mais en taille $n$ il y a $n!$ affectations : **23 pour $n=4$... mais 362 880 pour $n=9$**. Énumérer n'est pas prouver : il faut un **certificat**, dont la taille ne croît pas factoriellement. C'est tout l'objet du lake.

---
## 4. Le couple dual (`Assignment.DualFeasible`, `Assignment.dualValue`)

La lecture LP du problème (GT-27 section 3) associe à chaque ligne $i$ un potentiel $u_i$ et à chaque colonne $j$ un potentiel $v_j$. Le couple est **dual-réalisable** si $u_i + v_j \leq C_{ij}$ pour toute paire — ce sont exactement les *labels* de l'algorithme hongrois. Sa valeur duale est $\sum_i u_i + \sum_j v_j$ :

In [5]:
-- Un couple dual pour C3 (celui que la methode hongroise produit en terminant)
def u3 : Fin 3 → ℤ := ![1, 0, 0]
def v3 : Fin 3 → ℤ := ![2, 0, 2]

#eval Assignment.dualValue u3 v3

-- Un couple dual pour C3 (celui que la methode hongroise produit en terminant)
def u3 : Fin 3 → ℤ := ![1, 0, 0]
def v3 : Fin 3 → ℤ := ![2, 0, 2]

#eval Assignment.dualValue u3 v3
─────▶  5
--% env 4

Raw input:
{"cmd": "-- Un couple dual pour C3 (celui que la methode hongroise produit en terminant)\ndef u3 : Fin 3 \u2192 \u2124 := ![1, 0, 0]\ndef v3 : Fin 3 \u2192 \u2124 := ![2, 0, 2]\n\n#eval Assignment.dualValue u3 v3", "env": 3}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 5},
   "data": "5"}],
 "env": 4}

### Lecture du résultat

La valeur duale vaut **5** — exactement la valeur de $\sigma^*$. Cette coïncidence n'en est pas une : c'est le **gap de dualité nul**, le certificat que la méthode hongroise produit en terminant. Encore faut-il prouver que 5 est *inévitable* des deux côtés — c'est la dualité faible.

---
## 5. Dualité faible (`Assignment.weak_duality`) : le plancher

**Théorème (dualité faible)** — pour tout couple dual-réalisable $(u, v)$ et **toute** affectation $\sigma$ :

$$\text{dualValue}(u,v) \;\leq\; \text{value}(\sigma)$$

Aucune affectation ne peut descendre sous la valeur duale. La preuve du lake réindexe $\sum_j v_j$ le long de la permutation (un matching parfait visite chaque colonne exactement une fois) puis majore terme à terme par réalisabilité duale. Vérifions qu'elle ne repose sur rien de caché :

In [6]:
#print axioms Assignment.weak_duality

#print axioms Assignment.weak_duality
──────▶  'Assignment.weak_duality' depends on axioms: [propext, Classical.choice, Quot.sound]
--% env 5

Raw input:
{"cmd": "#print axioms Assignment.weak_duality", "env": 4}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "'Assignment.weak_duality' depends on axioms: [propext, Classical.choice, Quot.sound]"}],
 "env": 5}

### Lecture du résultat

`[propext, Classical.choice, Quot.sound]` — le **triple standard** : uniquement les axiomes logiques de Lean/Mathlib, aucun axiome mathématique ajouté, aucun `sorry`. La preuve est close par le noyau.

---
## 6. Le certificat à gap nul (`Assignment.optimality_of_zero_gap`)

Si $\sigma$ et $(u,v)$ atteignent le même bord — valeur primale = valeur duale — alors $\sigma$ est **optimale** : la dualité faible rend impossible tout $\tau$ strictement meilleur. Le lake décompose ce certificat en deux pas :

1. `Assignment.dualValue_eq_of_edges` — si **toutes les arêtes du matching sont des arêtes d'égalité** ($u_i + v_{\sigma(i)} = C_{i,\sigma(i)}$, définition `Assignment.EqEdge`), les valeurs primale et duale coïncident ;
2. `Assignment.optimality_of_zero_gap` — dual réalisable + valeurs égales ⇒ `Assignment.IsOptimal`.

Vérifions les trois arêtes de $\sigma^*$ sur notre fil rouge :

In [7]:
-- Les 3 aretes du matching sigma* = (0 <-> 1) sont des aretes d'egalite
example : Assignment.EqEdge C3 u3 v3 0 1 := by
  show u3 0 + v3 1 = C3 0 1; decide
example : Assignment.EqEdge C3 u3 v3 1 0 := by
  show u3 1 + v3 0 = C3 1 0; decide
example : Assignment.EqEdge C3 u3 v3 2 2 := by
  show u3 2 + v3 2 = C3 2 2; decide

#print axioms Assignment.dualValue_eq_of_edges
#print axioms Assignment.optimality_of_zero_gap

-- Les 3 aretes du matching sigma* = (0 <-> 1) sont des aretes d'egalite
example : Assignment.EqEdge C3 u3 v3 0 1 := by
  show u3 0 + v3 1 = C3 0 1; decide
example : Assignment.EqEdge C3 u3 v3 1 0 := by
  show u3 1 + v3 0 = C3 1 0; decide
example : Assignment.EqEdge C3 u3 v3 2 2 := by
  show u3 2 + v3 2 = C3 2 2; decide

#print axioms Assignment.dualValue_eq_of_edges
──────▶  'Assignment.dualValue_eq_of_edges' depends on axioms: [propext, Classical.choice, Quot.sound]
#print axioms Assignment.optimality_of_zero_gap
──────▶  'Assignment.optimality_of_zero_gap' depends on axioms: [propext, Classical.choice, Quot.sound]
--% env 6

Raw input:
{"cmd": "-- Les 3 aretes du matching sigma* = (0 <-> 1) sont des aretes d'egalite\nexample : Assignment.EqEdge C3 u3 v3 0 1 := by\n  show u3 0 + v3 1 = C3 0 1; decide\nexample : Assignment.EqEdge C3 u3 v3 1 0 := by\n  show u3 1 + v3 0 = C3 1 0; decide\nexample : Assignment.EqEdge C3 u3 v3 2 2 := by\n  show u3 2 + v3 2 = C3 2 2; decide\n\n#print axioms Assignment.dualValue_eq_of_edges\n#print axioms Assignment.optimality_of_zero_gap", "env": 5}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 9, "column": 0},
   "endPos": {"line": 9, "column": 6},
   "data":
   "'Assignment.dualValue_eq_of_edges' depends on axioms: [propext, Classical.choice, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 10, "column": 0},
   "endPos": {"line": 10, "column": 6},
   "data":
   "'Assignment.optimality_of_zero_gap' depends on axioms: [propext, Classical.choice, Quot.sound]"}],
 "env": 6}

### Lecture du résultat

Trois `decide` passent : $u_0 + v_1 = 1 + 0 = 1 = C_{01}$, $u_1 + v_0 = 0 + 2 = 2 = C_{10}$, $u_2 + v_2 = 0 + 2 = 2 = C_{22}$. Les deux théorèmes du certificat reposent sur le triple standard uniquement.

---
## 7. La charpente de correction (`Assignment.KuhnMunkres`)

Le dernier module assemble l'**invariant de sortie** de l'algorithme et son **étape de progression** :

- `Assignment.kuhn_munkres_correct` — dual réalisable + toutes les arêtes du matching dans le graphe d'égalité ⇒ optimal. C'est l'assemblage exact des deux sections précédentes : c'est ce que la méthode hongroise garantit **en terminant**, sans jamais consulter un solveur extérieur ;
- `Assignment.dualFeasible_tighten` — le **resserrement hongrois** ($u_i \mathrel{+}= \delta$ sur des lignes, $v_j \mathrel{-}= \delta$ sur des colonnes) préserve la réalisabilité duale, sous l'hypothèse que $\delta$ ne dépasse la marge d'aucune arête sortante. C'est l'étape qui, répétée, fait croître le graphe d'égalité jusqu'à permettre l'augmentation.

In [8]:
#print axioms Assignment.kuhn_munkres_correct
#print axioms Assignment.dualFeasible_tighten

#print axioms Assignment.kuhn_munkres_correct
──────▶  'Assignment.kuhn_munkres_correct' depends on axioms: [propext, Classical.choice, Quot.sound]
#print axioms Assignment.dualFeasible_tighten
──────▶  'Assignment.dualFeasible_tighten' depends on axioms: [propext, Classical.choice, Quot.sound]
--% env 7

Raw input:
{"cmd": "#print axioms Assignment.kuhn_munkres_correct\n#print axioms Assignment.dualFeasible_tighten", "env": 6}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "'Assignment.kuhn_munkres_correct' depends on axioms: [propext, Classical.choice, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "'Assignment.dualFeasible_tighten' depends on axioms: [propext, Classical.choice, Quot.sound]"}],
 "env": 7}

---
## 8. Le certificat complet : optimalité prouvée par le noyau

Assemblons tout sur le fil rouge. Deux `decide` établissent les hypothèses (réalisabilité duale de $(u_3, v_3)$ pour $C_3$ ; arêtes toutes d'égalité), et `Assignment.kuhn_munkres_correct` **conclut** : $\sigma^*$ est optimale pour $C_3$ — une preuve vérifiée par le noyau Lean, sans énumération des 6 permutations (et qui passerait identique en taille $n$, là où l'énumération explose en $n!$) :

In [9]:
theorem C3_dual_feasible : Assignment.DualFeasible C3 u3 v3 := by
  show ∀ i j : Fin 3, u3 i + v3 j ≤ C3 i j; decide

theorem optimal_C3 : Assignment.IsOptimal C3 (Equiv.swap (0 : Fin 3) 1) :=
  Assignment.kuhn_munkres_correct C3 u3 v3 (Equiv.swap (0 : Fin 3) 1)
    C3_dual_feasible
    (by show ∀ i : Fin 3, u3 i + v3 ((Equiv.swap (0 : Fin 3) 1) i)
          = C3 i ((Equiv.swap (0 : Fin 3) 1) i); decide)

#print axioms optimal_C3

theorem C3_dual_feasible : Assignment.DualFeasible C3 u3 v3 := by
  show ∀ i j : Fin 3, u3 i + v3 j ≤ C3 i j; decide

theorem optimal_C3 : Assignment.IsOptimal C3 (Equiv.swap (0 : Fin 3) 1) :=
  Assignment.kuhn_munkres_correct C3 u3 v3 (Equiv.swap (0 : Fin 3) 1)
    C3_dual_feasible
    (by show ∀ i : Fin 3, u3 i + v3 ((Equiv.swap (0 : Fin 3) 1) i)
          = C3 i ((Equiv.swap (0 : Fin 3) 1) i); decide)

#print axioms optimal_C3
──────▶  'optimal_C3' depends on axioms: [propext, Classical.choice, Quot.sound]
--% env 8

Raw input:
{"cmd": "theorem C3_dual_feasible : Assignment.DualFeasible C3 u3 v3 := by\n  show \u2200 i j : Fin 3, u3 i + v3 j \u2264 C3 i j; decide\n\ntheorem optimal_C3 : Assignment.IsOptimal C3 (Equiv.swap (0 : Fin 3) 1) :=\n  Assignment.kuhn_munkres_correct C3 u3 v3 (Equiv.swap (0 : Fin 3) 1)\n    C3_dual_feasible\n    (by show \u2200 i : Fin 3, u3 i + v3 ((Equiv.swap (0 : Fin 3) 1) i)\n          = C3 i ((Equiv.swap (0 : Fin 3) 1) i); decide)\n\n#print axioms optimal_C3", "env": 7}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 10, "column": 0},
   "endPos": {"line": 10, "column": 6},
   "data":
   "'optimal_C3' depends on axioms: [propext, Classical.choice, Quot.sound]"}],
 "env": 8}

### Lecture du résultat

`optimal_C3` ne dépend d'**aucun axiome** au-delà du triple standard. Le certificat a une taille **linéaire en $n$** (une hypothèse par ligne, une par arête du matching) : c'est précisément ce que `scipy.optimize.linear_sum_assignment` ne peut pas vous donner — lui répond une affectation, le lake répond une **preuve**.

---
## 9. Le resserrement hongrois en action (`Assignment.dualFeasible_tighten`)

Sur un dual **trivial** $u = v = 0$ (réalisable car $C_3 \geq 0$), resserrons $S = \{0\}$ (lignes) contre $T = \{1\}$ (colonnes) avec $\delta = 3$ — le maximum admissible : les arêtes sortantes $(0, j)$ pour $j \notin T$ ont marges $C_{00} = 4$ et $C_{02} = 3$, donc $\delta$ ne peut excéder $3$. Le théorème du lake garantit que le couple resserré reste réalisable — et les `decide` vérifient ses hypothèses une à une :

In [10]:
-- Le couple trivial, realisable car tous les couts sont positifs
def u0 : Fin 3 → ℤ := ![0, 0, 0]
def v0 : Fin 3 → ℤ := ![0, 0, 0]

theorem u0v0_dual_feasible : Assignment.DualFeasible C3 u0 v0 := by
  show ∀ i j : Fin 3, u0 i + v0 j ≤ C3 i j; decide

-- Le couple resserre : u0 + 3 sur la ligne 0, v0 - 3 sur la colonne 1
def uTight : Fin 3 → ℤ := fun i => if i ∈ ({0} : Finset (Fin 3)) then u0 i + 3 else u0 i
def vTight : Fin 3 → ℤ := fun j => if j ∈ ({1} : Finset (Fin 3)) then v0 j - 3 else v0 j

example : Assignment.DualFeasible C3 uTight vTight :=
  Assignment.dualFeasible_tighten C3 u0 v0 {0} {1} u0v0_dual_feasible 3
    (by decide) (by decide)

#eval Assignment.dualValue uTight vTight

-- Le couple trivial, realisable car tous les couts sont positifs
def u0 : Fin 3 → ℤ := ![0, 0, 0]
def v0 : Fin 3 → ℤ := ![0, 0, 0]

theorem u0v0_dual_feasible : Assignment.DualFeasible C3 u0 v0 := by
  show ∀ i j : Fin 3, u0 i + v0 j ≤ C3 i j; decide

-- Le couple resserre : u0 + 3 sur la ligne 0, v0 - 3 sur la colonne 1
def uTight : Fin 3 → ℤ := fun i => if i ∈ ({0} : Finset (Fin 3)) then u0 i + 3 else u0 i
def vTight : Fin 3 → ℤ := fun j => if j ∈ ({1} : Finset (Fin 3)) then v0 j - 3 else v0 j

example : Assignment.DualFeasible C3 uTight vTight :=
  Assignment.dualFeasible_tighten C3 u0 v0 {0} {1} u0v0_dual_feasible 3
    (by decide) (by decide)

#eval Assignment.dualValue uTight vTight
─────▶  0
--% env 9

Raw input:
{"cmd": "-- Le couple trivial, realisable car tous les couts sont positifs\ndef u0 : Fin 3 \u2192 \u2124 := ![0, 0, 0]\ndef v0 : Fin 3 \u2192 \u2124 := ![0, 0, 0]\n\ntheorem u0v0_dual_feasible : Assignment.DualFeasible C3 u0 v0 := by\n  show \u2200 i j : Fin 3, u0 i + v0 j \u2264 C3 i j; decide\n\n-- Le couple resserre : u0 + 3 sur la ligne 0, v0 - 3 sur la colonne 1\ndef uTight : Fin 3 \u2192 \u2124 := fun i => if i \u2208 ({0} : Finset (Fin 3)) then u0 i + 3 else u0 i\ndef vTight : Fin 3 \u2192 \u2124 := fun j => if j \u2208 ({1} : Finset (Fin 3)) then v0 j - 3 else v0 j\n\nexample : Assignment.DualFeasible C3 uTight vTight :=\n  Assignment.dualFeasible_tighten C3 u0 v0 {0} {1} u0v0_dual_feasible 3\n    (by decide) (by decide)\n\n#eval Assignment.dualValue uTight vTight", "env": 8}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 16, "column": 0},
   "endPos": {"line": 16, "column": 5},
   "data": "0"}],
 "env": 9}

### Lecture du résultat

Le couple resserré reste dual-réalisable — et sa valeur duale a **crû** de 0 à 3 : le resserrement est le moteur de la montée duale de l'algorithme (GT-27 section 4), et `dualFeasible_tighten` en est la garantie de sûreté, prouvée une fois pour toutes.

---
## 10. Exercices

Les trois exercices suivent la progression du notebook. À compléter **dans une copie** des cellules — les stubs ne lèvent pas d'erreur, le notebook reste exécutable de bout en bout.

In [11]:
-- Exercice 1 (Definitions) : votre propre matrice 2x2.
-- Construire une matrice C2 : Fin 2 → Fin 2 → ℤ dont l'affectation optimale
-- N'EST PAS l'identite (indices defies : la matrice [[1, 0], [0, 1]] repond id).
--
-- Etape 1 : definir C2 avec la notation ![![.., ..], ![.., ..]].
-- Etape 2 : #eval Assignment.value C2 sur Equiv.refl _ et (Equiv.swap (0 : Fin 2) 1).
-- Etape 3 : verifier que la transposition fait strictement mieux.
--
-- def C2 : Fin 2 → Fin 2 → ℤ := ![![..], ![..]]  -- TODO etudiant
example : True := trivial

-- Exercice 1 (Definitions) : votre propre matrice 2x2.
-- Construire une matrice C2 : Fin 2 → Fin 2 → ℤ dont l'affectation optimale
-- N'EST PAS l'identite (indices defies : la matrice [[1, 0], [0, 1]] repond id).
--
-- Etape 1 : definir C2 avec la notation ![![.., ..], ![.., ..]].
-- Etape 2 : #eval Assignment.value C2 sur Equiv.refl _ et (Equiv.swap (0 : Fin 2) 1).
-- Etape 3 : verifier que la transposition fait strictement mieux.
--
-- def C2 : Fin 2 → Fin 2 → ℤ := ![![..], ![..]]  -- TODO etudiant
example : True := trivial
--% env 10

Raw input:
{"cmd": "-- Exercice 1 (Definitions) : votre propre matrice 2x2.\n-- Construire une matrice C2 : Fin 2 \u2192 Fin 2 \u2192 \u2124 dont l'affectation optimale\n-- N'EST PAS l'identite (indices defies : la matrice [[1, 0], [0, 1]] repond id).\n--\n-- Etape 1 : definir C2 avec la notation ![![.., ..], ![.., ..]].\n-- Etape 2 : #eval Assignment.value C2 sur Equiv.refl _ et (Equiv.swap (0 : Fin 2) 1).\n-- Etape 3 : verifier que la transposition fait strictement mieux.\n--\n-- def C2 : Fin 2 \u2192 Fin 2 \u2192 \u2124 := ![![..], ![..]]  -- TODO etudiant\nexample : True := trivial", "env": 9}
Raw output:
{"env": 10}

In [12]:
-- Exercice 2 (Duality + Optimality) : le certificat de votre matrice.
-- Pour votre C2 de l'Exercice 1, trouver un couple dual (u2, v2) a gap nul
-- avec l'affectation optimale sigma2 trouvee ci-dessus, puis le PROUVER :
--
-- Etape 1 : deviner u2, v2 (sommes egales a la valeur optimale, u_i + v_j <= C2 i j partout).
-- Indice : partir de u = v = 0 et resserrer, comme en section 9.
-- Etape 2 : theorem C2_dual_feasible : Assignment.DualFeasible C2 u2 v2 := by decide
-- Etape 3 : theorem optimal_C2 : Assignment.IsOptimal C2 (Equiv.swap (0 : Fin 2) 1 :=
--   Assignment.kuhn_munkres_correct C2 u2 v2 _ C2_dual_feasible (by decide)
--
-- TODO etudiant
example : True := trivial

-- Exercice 2 (Duality + Optimality) : le certificat de votre matrice.
-- Pour votre C2 de l'Exercice 1, trouver un couple dual (u2, v2) a gap nul
-- avec l'affectation optimale sigma2 trouvee ci-dessus, puis le PROUVER :
--
-- Etape 1 : deviner u2, v2 (sommes egales a la valeur optimale, u_i + v_j <= C2 i j partout).
-- Indice : partir de u = v = 0 et resserrer, comme en section 9.
-- Etape 2 : theorem C2_dual_feasible : Assignment.DualFeasible C2 u2 v2 := by decide
-- Etape 3 : theorem optimal_C2 : Assignment.IsOptimal C2 (Equiv.swap (0 : Fin 2) 1 :=
--   Assignment.kuhn_munkres_correct C2 u2 v2 _ C2_dual_feasible (by decide)
--
-- TODO etudiant
example : True := trivial
--% env 11

Raw input:
{"cmd": "-- Exercice 2 (Duality + Optimality) : le certificat de votre matrice.\n-- Pour votre C2 de l'Exercice 1, trouver un couple dual (u2, v2) a gap nul\n-- avec l'affectation optimale sigma2 trouvee ci-dessus, puis le PROUVER :\n--\n-- Etape 1 : deviner u2, v2 (sommes egales a la valeur optimale, u_i + v_j <= C2 i j partout).\n-- Indice : partir de u = v = 0 et resserrer, comme en section 9.\n-- Etape 2 : theorem C2_dual_feasible : Assignment.DualFeasible C2 u2 v2 := by decide\n-- Etape 3 : theorem optimal_C2 : Assignment.IsOptimal C2 (Equiv.swap (0 : Fin 2) 1 :=\n--   Assignment.kuhn_munkres_correct C2 u2 v2 _ C2_dual_feasible (by decide)\n--\n-- TODO etudiant\nexample : True := trivial", "env": 10}
Raw output:
{"env": 11}

In [13]:
-- Exercice 3 (KuhnMunkres) : le resserrement maximal sur la ligne 1.
-- Partir du couple trivial u0 = v0 = 0 (section 9) et resserrer S = {1} avec T = ∅ :
-- les aretes sortantes sont (1, 0), (1, 1), (1, 2) de marges 2, 0, 5.
--
-- Etape 1 : quel est le delta maximal admissible ? (reponse attendue : 0 —
-- la marge nulle de l'arete (1, 1) plafonne le resserrement).
-- Etape 2 : le verifier en executant dualFeasible_tighten avec ce delta
-- (structure de la section 9, avec S = {1} et T = ∅) :
-- example : DualFeasible C3 uTightBis vTightBis :=
--   Assignment.dualFeasible_tighten C3 u0 v0 {1} ∅ (by decide) .. (by decide) (by decide)
-- Etape 3 : expliquer (en markdown) pourquoi une marge nulle bloque la montee duale,
-- et ce que l'algorithme fait alors (indice : GT-27 section 4, colonne entree dans T).
--
-- TODO etudiant
example : True := trivial

-- Exercice 3 (KuhnMunkres) : le resserrement maximal sur la ligne 1.
-- Partir du couple trivial u0 = v0 = 0 (section 9) et resserrer S = {1} avec T = ∅ :
-- les aretes sortantes sont (1, 0), (1, 1), (1, 2) de marges 2, 0, 5.
--
-- Etape 1 : quel est le delta maximal admissible ? (reponse attendue : 0 —
-- la marge nulle de l'arete (1, 1) plafonne le resserrement).
-- Etape 2 : le verifier en executant dualFeasible_tighten avec ce delta
-- (structure de la section 9, avec S = {1} et T = ∅) :
-- example : DualFeasible C3 uTightBis vTightBis :=
--   Assignment.dualFeasible_tighten C3 u0 v0 {1} ∅ (by decide) .. (by decide) (by decide)
-- Etape 3 : expliquer (en markdown) pourquoi une marge nulle bloque la montee duale,
-- et ce que l'algorithme fait alors (indice : GT-27 section 4, colonne entree dans T).
--
-- TODO etudiant
example : True := trivial
--% env 12

Raw input:
{"cmd": "-- Exercice 3 (KuhnMunkres) : le resserrement maximal sur la ligne 1.\n-- Partir du couple trivial u0 = v0 = 0 (section 9) et resserrer S = {1} avec T = \u2205 :\n-- les aretes sortantes sont (1, 0), (1, 1), (1, 2) de marges 2, 0, 5.\n--\n-- Etape 1 : quel est le delta maximal admissible ? (reponse attendue : 0 \u2014\n-- la marge nulle de l'arete (1, 1) plafonne le resserrement).\n-- Etape 2 : le verifier en executant dualFeasible_tighten avec ce delta\n-- (structure de la section 9, avec S = {1} et T = \u2205) :\n-- example : DualFeasible C3 uTightBis vTightBis :=\n--   Assignment.dualFeasible_tighten C3 u0 v0 {1} \u2205 (by decide) .. (by decide) (by decide)\n-- Etape 3 : expliquer (en markdown) pourquoi une marge nulle bloque la montee duale,\n-- et ce que l'algorithme fait alors (indice : GT-27 section 4, colonne entree dans T).\n--\n-- TODO etudiant\nexample : True := trivial", "env": 11}
Raw output:
{"env": 12}

---
## Conclusion

Le lake `assignment_lean` donne au triple test de GT-27 son statut logique complet : la **dualité faible** fixe le plancher, le **gap nul** le certificat, l'**invariant de sortie** la garantie de terminaison correcte, le **resserrement** la sûreté de chaque étape. Là où `scipy.optimize.linear_sum_assignment` rend une affectation, le lake rend une **preuve vérifiée par le noyau** — et le compagnon 27b est le lieu où ces preuves s'exécutent et se lisent.

Ce qui reste délibérément hors scope du lake (cf son README) : la preuve de **terminaison** et la complexité $O(n^3)$ (Edmonds-Karp/Tomizawa), et le pont **Shapley-Shubik** (cœur du jeu d'affectation = solutions duales optimales), traité numériquement dans GT-27 section 5.

## Références

- Lake : [`assignment_lean/`](assignment_lean/) — README (charpente, hors-scope, build), issue #12598 (hommage Munkres 1930–2026)
- Kuhn (1955), *The Hungarian Method for the Assignment Problem*, Naval Research Logistics Quarterly 2:83–97 ; Munkres (1957), *Algorithms for the Assignment and Transportation Problems*, J. SIAM 5(1):32–38
- Implémentation SOTA : `scipy.optimize.linear_sum_assignment` (GT-27 section 2)
- EPIC #11703 — visibilité des lakes Lean dans les notebooks (ce compagnon est le volet natif de `assignment_lean`)